# RetailPulse: Data Cleaning and Preparation

## Overview

This notebook focuses on cleaning and preparing the Amazon Sale Report dataset for reliable analysis. It applies data quality findings from the previous notebook to address missing values, inconsistent categorical data, data types, and other identified issues.

## Objectives

* Create a working copy of the raw dataset for cleaning.
* Remove columns identified as unnecessary for the planned analysis.
* Handle missing values using appropriate strategies.
* Standardize categorical values and correct data types where needed.
* Investigate unusual values and validate the cleaned dataset.
* Save the cleaned dataset for subsequent analysis.

## Approach

The original raw dataset will remain unchanged. All cleaning operations will be performed on a separate working DataFrame, and the results will be validated before saving the processed dataset.


## 1. Load the Dataset

The raw Amazon Sale Report dataset is loaded into a Pandas DataFrame. A separate working copy is created to ensure that all cleaning operations leave the original dataset unchanged.


In [88]:
import pandas as pd
import numpy as np

csv_file = "../data/raw/Amazon Sale Report.csv"

df = pd.read_csv(csv_file)

df_cleaned = df.copy()

print("Dataset loaded successfully.")
print(f"Original dataset shape: {df.shape}")
print(f"Working copy shape: {df_cleaned.shape}")

Dataset loaded successfully.
Original dataset shape: (128975, 24)
Working copy shape: (128975, 24)


C:\Users\praya\AppData\Local\Temp\ipykernel_12844\4064105777.py:6: DtypeWarning: Columns (23: Unnamed: 22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)


## 2. Standardize Column Names

This section standardizes column names by removing leading and trailing whitespace and replacing spaces with underscores. This creates consistent, easy-to-reference column names for subsequent cleaning and analysis.


In [89]:
df_cleaned.columns = (
    df_cleaned.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Columns names are standardized successfully.")
df_cleaned.columns.to_list()

Columns names are standardized successfully.


['index',
 'order_id',
 'date',
 'status',
 'fulfilment',
 'sales_channel',
 'ship_service_level',
 'style',
 'sku',
 'category',
 'size',
 'asin',
 'courier_status',
 'qty',
 'currency',
 'amount',
 'ship_city',
 'ship_state',
 'ship_postal_code',
 'ship_country',
 'promotion_ids',
 'b2b',
 'fulfilled_by',
 'unnamed:_22']

## 3. Remove Unnecessary Columns

This section removes columns identified during the data quality assessment as unnecessary for the planned analysis. The selected columns are removed from the working DataFrame, while the original raw dataset remains unchanged.


In [90]:
drop_columns = [
    "index",
    "currency",
    "ship_city",
    "ship_postal_code",
    "promotion_ids",
    "unnamed:_22"
]

drop_columns

['index',
 'currency',
 'ship_city',
 'ship_postal_code',
 'promotion_ids',
 'unnamed:_22']

In [91]:
df_cleaned.drop(columns=drop_columns, inplace=True)

In [92]:
print(f"Columns removed: {len(drop_columns)}")
print(f"Remaining columns: {df_cleaned.shape[1]}")

df_cleaned.columns.to_list()

Columns removed: 6
Remaining columns: 18


['order_id',
 'date',
 'status',
 'fulfilment',
 'sales_channel',
 'ship_service_level',
 'style',
 'sku',
 'category',
 'size',
 'asin',
 'courier_status',
 'qty',
 'amount',
 'ship_state',
 'ship_country',
 'b2b',
 'fulfilled_by']

## 4. Investigate Duplicate Order IDs

An order ID may appear in multiple rows when an order contains multiple products or line items. Therefore, repeated order IDs do not necessarily indicate duplicate records.

This section identifies repeated order IDs and examines their frequency before deciding how to handle them. Valid line items will be preserved to avoid losing sales data.


In [93]:
print("Duplicate order ids: ", df_cleaned["order_id"].duplicated().sum())

print("Exact duplicate records along dataset: ", df_cleaned.duplicated().sum())


Duplicate order ids:  8597
Exact duplicate records along dataset:  6


In [94]:
duplicates_before = df_cleaned.duplicated().sum()

df_cleaned.drop_duplicates(inplace=True)

duplicates_after = df_cleaned.duplicated().sum()

print(f"Exact duplicate records removed: {duplicates_before - duplicates_after}")
print(f"Exact duplicate records remaining: {duplicates_after}")
print(f"Updated dataset shape: {df_cleaned.shape}")

Exact duplicate records removed: 6
Exact duplicate records remaining: 0
Updated dataset shape: (128969, 18)


## 5. Convert the Date Column

The `date` column represents the date associated with each sales record. Converting it to a datetime data type enables accurate date-based filtering, time-series analysis, and the calculation of daily, monthly, and yearly sales trends.

This section converts the date values to a datetime format and checks for any values that could not be converted successfully.


In [95]:
df_cleaned["date"] = pd.to_datetime(
    df_cleaned["date"],
    format="%m-%d-%y",
    errors="coerce"
)

print("Date column converted successfully.")
print(f"Data type: {df_cleaned['date'].dtype}")
print(f"Missing dates: {df_cleaned['date'].isna().sum()}")
print(f"Date range: {df_cleaned['date'].min()} to {df_cleaned['date'].max()}")

Date column converted successfully.
Data type: datetime64[us]
Missing dates: 0
Date range: 2022-03-31 00:00:00 to 2022-06-29 00:00:00


## 6. Handle Missing Courier Status Values

The `courier_status` column contains missing values that need to be addressed for consistent shipment analysis. These missing values will be replaced with `Unshipped` to provide a consistent category for subsequent analysis.


In [123]:
df_cleaned["courier_status"] = df_cleaned["courier_status"].fillna("Unshipped")

print(f"Missing values after cleaning: {df_cleaned["courier_status"].isna().sum()}")
print("\nCourier status value counts:")
print(df_cleaned["courier_status"].value_counts())

Missing values after cleaning: 0

Courier status value counts:
courier_status
Shipped      109484
Unshipped     13553
Cancelled      5932
Name: count, dtype: int64


## 7. Handle Missing Amount Values

The `amount` column represents the monetary value associated with each sales record. Missing values are replaced with `0` to maintain consistent data for subsequent calculations and analysis.

This approach assumes that missing amount values should be treated as zero for this project. Missing values and genuine zero-value transactions will be considered separately during the final inspection.


In [124]:
df_cleaned["amount"] = df_cleaned["amount"].fillna("0")

print(f"Missing values after cleaning: {df_cleaned["amount"].isna().sum()}")
print("\nAmount value counts:")
print(df_cleaned["amount"].value_counts())

Missing values after cleaning: 0

Amount value counts:
amount
0         7792
399.0     5442
771.0     2797
735.0     2436
0.0       2343
          ... 
719.05       1
829.52       1
708.58       1
1244.0       1
639.0        1
Name: count, Length: 1411, dtype: int64


## 8. Handle Missing Shipping Country Values

The `ship_country` column identifies the destination country for each sales record. During the data quality assessment, all non-missing values were found to be `IN`, while some records contained missing values.

Since the available records consistently identify India as the shipping country, missing values will be replaced with `IN` to maintain consistency in the dataset.


In [137]:
df_cleaned["ship_country"] = df_cleaned["ship_country"].fillna("0")

print(f"Missing values after cleaning: {df_cleaned["ship_country"].isna().sum()}")
print("\nShipping Country value counts:")
print(df_cleaned["ship_country"].value_counts())

Missing values after cleaning: 0

Shipping Country value counts:
ship_country
IN    128969
Name: count, dtype: int64


## 9. Handle Missing Fulfilment Values

The `fulfilled_by` column identifies the fulfilment method associated with a sales record. The data quality assessment found that a substantial proportion of values are missing, while the available records contain the category `Easy Ship`.

Since the missing values cannot be reliably assigned to a known fulfilment method, they will be replaced with `Unknown`. This preserves the existing information while clearly identifying records with unavailable fulfilment details.


In [135]:
df_cleaned["fulfilled_by"] = df_cleaned["fulfilled_by"].fillna("Unknown")

print(f"Missing values after cleaning: {df_cleaned["fulfilled_by"].isna().sum()}")
print("\nFullfilment values countd:")
print(df_cleaned["fulfilled_by"].value_counts())

Missing values after cleaning: 0

Fullfilment values countd:
fulfilled_by
Unknown      89692
Easy Ship    39277
Name: count, dtype: int64


## 10. Standardize B2B Values

The `b2b` column identifies whether a sales record is associated with a business-to-business (B2B) transaction. Its values are currently represented as Boolean values, `True` and `False`.

To improve readability and make the categories easier to interpret during analysis, `True` will be replaced with `Business` and `False` with `Customer`.


In [148]:
df_cleaned["b2b"] = df_cleaned["b2b"].replace({True: "Business", False: "Customer"})

print("B2B values standardized successfully.")
print("\nB2B value counts:")
print(df_cleaned["b2b"].value_counts())

B2B values standardized successfully.

B2B value counts:
b2b
Customer    128098
Business       871
Name: count, dtype: int64
